In [ ]:
%matplotlib widget

In [ ]:
from glob import glob
import numpy as np
import pandas as pd
import flammkuchen as fl
from split_dataset import SplitDataset
from bouter import Experiment
from motions.utilities import stim_vel_dir_dataframe, quantize_directions
from scipy.interpolate import interp1d 
from scipy.signal import convolve2d
import colorspacious
import napari
import matplotlib.pyplot as plt

from fimpylab.core.twop_experiment import TwoPExperiment

from pathlib import Path

In [ ]:
'''
# calculate dot product with each regressor from dF/F traces, px-wise
def get_tuning_map(img, sens_regs):
    traces = img.reshape(img.shape[0], -1)

    n_t = sens_regs.shape[0]
    a = sens_regs @ traces - sens_regs.shape[1] * np.nanmean(sens_regs, 1) * np.nanmean(traces)
    b = (sens_regs.shape[1] - 1) * np.nanstd(traces, 0) * np.nanstd(sens_regs)
    reg = a / b
    reg = reg.reshape(reg.shape[0], img.shape[-2], img.shape[-1])

    return reg
'''

In [ ]:
# calculate dot product with each regressor from dF/F traces, px-wise
def get_tuning_map(img, sens_regs):
    traces = img.reshape(img.shape[0], -1)
    n_t = sens_regs.shape[0]
    print(np.shape(sens_regs))
    print(np.shape(traces))
    
    traces = traces[:n_t,:]
    a = np.dot(traces.T, sens_regs) - traces.shape[0] * np.outer(np.nanmean(traces, 0), np.nanmean(sens_regs, 0))
    b = (traces.shape[0] - 1) * np.outer(np.nanstd(traces, 0), np.nanstd(sens_regs, 0))
    reg = (a / b).T
    reg = reg.reshape(reg.shape[0], img.shape[-2], img.shape[-1])

    return reg

In [ ]:
master = Path(r"Z:\Hagar and Ot\E0040\v10\2p\s1186t")
fish_list = list(master.glob("*_f*"))
fish = fish_list[2]
print(fish)

aligned = SplitDataset(fish / "aligned")
exp_list = glob(str(fish / "behavior/*.json"))

sampling = 1/3
time = np.linspace(0, aligned.shape[0]*sampling, aligned.shape[0])
n_dir = 8

In [ ]:
len_rec, num_planes, x_pix, y_pix = np.shape(aligned)
np.shape(aligned)

In [ ]:
stack = SplitDataset(fish / "dff")

In [ ]:
# choose plane and calculate correlation map with chosen regressor 
plane_list = glob(str(fish / "suite2p/00*"))


In [ ]:
np.shape(regs)

In [ ]:
plane_corr = np.zeros((num_planes, n_dir, x_pix, y_pix ))
for i in range(num_planes):
    plane = plane_list[i]
    print(plane)
    
    file_name = "sensory_regressors" + str(i) + ".h5"
    regs = fl.load(fish / file_name)['regressors'].values
    
    plane_corr[i] = get_tuning_map(stack[:,i,:,:], regs)

In [ ]:
np.shape(plane_corr)

In [ ]:
d = {'plane_corr': plane_corr}
fl.save(fish / 'plane_corrmap_corrvalues.h5', d)

In [ ]:
np.min(plane_corr)

In [ ]:
n_dir = 8
sampling = 1/3

In [ ]:
for fish in fish_list:
    print(fish)
    if not (fish / "plane_corrmap_corrvalues.h5").exists():
        #try:
        stack = SplitDataset(fish / "dff")
        exp_list = glob(str(fish / "behavior/*.json"))
        time = np.linspace(0, stack.shape[0]*sampling, stack.shape[0])

        len_rec, num_planes, x_pix, y_pix = np.shape(stack)

        plane_list = glob(str(fish / "suite2p/00*"))

        plane_corr = np.zeros((num_planes, n_dir, x_pix, y_pix ))
        for i in range(num_planes):
            plane = plane_list[i]
            print(plane)

            file_name = "sensory_regressors" + str(i) + ".h5"
            regs = fl.load(fish / file_name)['regressors'].values

            plane_corr[i] = get_tuning_map(stack[:,i,:,:], regs)

        d = {'plane_corr': plane_corr}
        fl.save(fish / 'plane_corrmap_corrvalues.h5', d)
        #except:
        #    print("Error")

In [ ]:
import napari
%gui qt5

In [ ]:
plane_corr = fl.load(fish / 'plane_corrmap_corrvalues.h5')['plane_corr']

In [ ]:
with napari.gui_qt():
    v = napari.view_image(plane_corr)